In [1]:
import os
import gc
import re
import math
import json
import time
import pandas as pd
import torch
import ollama
from PIL import Image as PILImage
from datetime import datetime
from pathlib import Path
from typing import List, Dict
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_compressors import FlashrankRerank
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from ultralytics import YOLO, YOLOWorld
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Nirvana\anaconda3\envs\torch_fast\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Nirvana\anaconda3\envs\torch_fast\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Nirvana\anaconda3\envs\torch_fast\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\Nirvana\anaconda3\envs\torch_fast\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xad in position 7: invalid start byte


In [2]:
data_to_process = pd.read_excel(r'AI_agent_FASE\preparing_data.xlsx')
data_to_process = data_to_process.where(pd.notnull(data_to_process), None).drop(columns=['Unnamed: 0'])

In [ ]:
class FSAEHybridRAG:

    def __init__(self, pdf_path: str, db_dir: str = "./fsae_chroma"):

        self.db_dir = db_dir

        self.embeddings = HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True}
        )

        self.rule_documents = self.parse_rulebook(pdf_path)


        if os.path.exists(db_dir):
            self.vector_db = Chroma(persist_directory=db_dir, embedding_function=self.embeddings)

        else:

            self.vector_db = Chroma.from_documents(
                documents=self.rule_documents,
                embedding=self.embeddings,
                persist_directory=db_dir
            )

        self.bm25_retriever = BM25Retriever.from_documents(self.rule_documents)

    def parse_rulebook(self, pdf_path: str) -> List[Document]:

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        full_text = "\n".join([p.page_content for p in pages[4:]])

        rule_pattern = re.compile(
            r"^\s*(?P<rule>[A-Z]{1,2}\.\d+(?:\.\d+){0,2}[a-zA-Z]?)\s+(?P<title>[^\n]+)",
            re.MULTILINE
        )

        matches = list(rule_pattern.finditer(full_text))

        documents = []

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=200
        )

        for i, match in enumerate(matches):
            start = match.start()

            if i + 1 < len(matches):
                end = matches[i + 1].start()
            else:
                end = len(full_text)

            rule_text = full_text[start:end].strip()
            if len(rule_text) < 50:
                continue

            rule_id = match.group("rule")
            title = match.group("title").strip()

            chunks = splitter.split_text(rule_text)

            for chunk_id, chunk in enumerate(chunks):

                full_chunk_text = f"rule {rule_id}: {title}\n{chunk}"

                doc = Document(
                    page_content=full_chunk_text,
                    metadata={
                        "rule_id": rule_id.lower(),
                        "title": title.lower(),
                        "parent_id": self.get_parent_id(rule_id.lower()),
                        "chunk_id": chunk_id
                    }
                )

                documents.append(doc)

        return documents
    
    def get_parent_id(self, rule_id: str):
        parts = rule_id.split('.')
        if len(parts) > 1:
            return ".".join(parts[:-1])
        return rule_id

    def extract_rule_id(self, query: str):
        pattern = re.compile(
            r"([A-Za-z]{1,2}\.\d+(?:\.\d+){0,2}[a-zA-Z]?)" 
        )
        matches = pattern.findall(query)
        if matches:
            return list(set([m.lower() for m in matches]))
        return None

    def exact_rule_search(self, rule_id: str):
        rule_id_clean = rule_id.lower()
            
        results = self.vector_db.similarity_search(
            query=rule_id_clean,
            k=15,
            filter={
                "$or": [
                    {"rule_id": rule_id_clean},
                    {"parent_id": rule_id_clean}
                ]
            }
        )
        
        found = any(doc.metadata.get("rule_id") == rule_id_clean for doc in results)
        
        if not found:
            parent_id = re.sub(r'[a-z]$', '', rule_id_clean)
            if parent_id != rule_id_clean:
                parent_results = self.vector_db.similarity_search(
                    query=parent_id,
                    k=10,
                    filter={"$or": [
                        {"rule_id": parent_id},
                        {"parent_id": parent_id}
                    ]}
                )
                if parent_results:
                    print(f" '{rule_id_clean}' не найден — использую родителя '{parent_id}'")
                    results = results + parent_results
                    found = True

        if not found:
            print(f" '{rule_id_clean}' не найден — BM25 по тексту")
            bm25_results = self.bm25_retriever.invoke(rule_id_clean)
            results = results + bm25_results

        return results


    def vector_search(self, query: str, k=10):

        retriever = self.vector_db.as_retriever(
            search_kwargs={"k": k}
        )

        docs = retriever.invoke(query)

        return docs

    def bm25_search(self, query: str):

        return self.bm25_retriever.invoke(query)
    
    def _compute_rrf_scores(self, results_lists: list, k=60):
        scores = {}
        doc_map = {}

        for results in results_lists:
            for rank, doc in enumerate(results):
                key = doc.metadata.get("rule_id","") + str(doc.metadata.get("chunk_id",0))
                scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
                doc_map[key] = doc

        return scores, doc_map

    def reciprocal_rank_fusion(self, results_lists: list, k=60) -> list:
        scores, doc_map = self._compute_rrf_scores(results_lists, k)
        sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
        return [doc_map[key] for key in sorted_keys]

    def hybrid_search(self, query: str, k=5):
        rule_id = self.extract_rule_id(query)
        if rule_id:
            exact_docs = []
            for rid in rule_id:
                exact_docs.extend(self.exact_rule_search(rid))
            
            if exact_docs:
                unique_docs = {}
                for doc in exact_docs:
                    key = doc.metadata.get('rule_id', '') + str(doc.metadata.get('chunk_id', 0))
                    unique_docs[key] = doc
                return list(unique_docs.values())[:k]

        self.bm25_retriever.k = max(10, k * 2)
        bm25_docs = self.bm25_search(query)
        vector_docs = self.vector_search(query, k=k * 2)

        rrf_scores, doc_map = self._compute_rrf_scores([bm25_docs, vector_docs])

        top_rrf_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k * 3]
        top_rrf_docs = [doc_map[key] for key in top_rrf_keys]

        compressor = FlashrankRerank(top_n=k)
        reranked = compressor.compress_documents(top_rrf_docs, query)

        reranked_scores = {}
        for rank, doc in enumerate(reranked):
            key = doc.metadata.get("rule_id","") + str(doc.metadata.get("chunk_id",0))
            reranked_scores[key] = 1.0 / (1 + math.log1p(rank))

        max_rrf = max(rrf_scores.values()) if rrf_scores else 1
        normalized_rrf = {key: v / max_rrf for key, v in rrf_scores.items()}

        final_scores = {}
        all_keys = set(list(rrf_scores.keys()) + list(reranked_scores.keys()))
        for key in all_keys:
            final_scores[key] = (
                normalized_rrf.get(key, 0) * 0.6 +
                reranked_scores.get(key, 0) * 0.4
            )

        def get_key(docs):
            if not docs:
                return None
            d = docs[0]
            return d.metadata.get("rule_id","") + str(d.metadata.get("chunk_id",0))

        bm25_top_key = get_key(bm25_docs)
        vector_top_key = get_key(vector_docs)
        if bm25_top_key and bm25_top_key == vector_top_key:
            final_scores[bm25_top_key] = final_scores.get(bm25_top_key, 0) + 0.3

        sorted_keys = sorted(final_scores, key=final_scores.get, reverse=True)
        return [doc_map[key] for key in sorted_keys if key in doc_map][:k]

    def get_context(self, query: str, k: int = 5):

        docs = self.hybrid_search(query, k=k)

        result = []

        for doc in docs:

            text = f"""
            RULE: {doc.metadata.get('rule_id')}
            TITLE: {doc.metadata.get('title')}

            CONTENT:
            {doc.page_content}
            """

            result.append(text)

        return result

class DecesionCenter:
    def __init__(self, model_name="gemma4:e4b"):
        self.model = model_name

    def keywords_generate(self, question, rag_response):
        prompt = f"""
            You are an FSAE Technical Scrutineer.
            Based on this question about the competition rules,
            name the keywords for the drawing elements that the YOLO model will need to find,
            so that you can analyze the drawing image more accurately in the future.
            In the answer, list no more than 5 words.
            KEYWORD RULES:
            - Keywords must be VISIBLE objects on a technical drawing (e.g. "dimension line", "wheel outline", "scale bar")
            - DO NOT use abstract words: "compliance", "rule", "requirement", "standard"
            - Maximum 5 keywords
            - DO NOT use the question itself as a keyword.
            The keywords will be used by YOLO to DRAW BOUNDING BOXES around parts.
            You can also refer to the RAG database, which contains vectorized rules and technical specifications.
            The RAG database contains ONLY the text of the FSAE Rules PDF. It CANNOT see images.
            The rules that RAG found for you, if there are several, are separated by the symbol " | "
            TECHNICAL CONTEXT FROM RULEBOOK:
            {self.format_rag_context(rag_response)}
            Also, in the course of further analysis, two different YOLO models can be used: YOLO-world and YOLOv26. 
            YOLO MODEL CHOICE:
            - "YOLO-world": better for large structural elements (chassis, roll hoops, bodywork)
            - "YOLOv26": better for small details (dimension arrows, numbers, labels, bolts)
            SAHI WINDOW SIZE:
            - Large window (6000-8000): when the object spans most of the drawing
            - Small window (640-4000): when looking for small annotations or labels
            - Overlap 0.5-0.8: large objects | Overlap 0.85-0.95: small objects requiring precision
            Question: "{question}"
            As an answer, output all these parameters in JSON format. 
            Example Output: 
            {{
                "detection_model": "YOLO-world" or "YOLOv26",
                "slice_height": integer,
                "slice_width": integer,
                "overlap_height_ratio": float,
                "overlap_width_ratio": float,
                "keywords": ["first keyword", "second keyword", "third keyword", "fourt keyword"]
            }}
            Return ONLY valid JSON. No additional text or explanations.
            CRITICAL: OUTPUT ONLY JSON. START YOUR RESPONSE WITH '{{' AND END WITH '}}'. 
            DO NOT WRITE MARKDOWN. DO NOT WRITE 'Here is the JSON'. 
            ONLY THE RAW JSON OBJECT.
            """
        
        response = ollama.generate(
            model=self.model, 
            prompt=prompt, 
            format="json",
            keep_alive=300, 
            options={"temperature": 0.3}
            )

        return response
    
    def format_rag_context(self, rag_list):
        cleaned_context = []
        for item in rag_list:
            clean_text = " ".join(item.split())
            cleaned_context.append(f"RULE INFO: {clean_text}")
        
        return " | ".join(cleaned_context)
    
    def database_query(self, orig_quest):

        prompt = f"""
            You are an FSAE Rules Expert. 
            Your goal is to formulate a search query that matches the EXACT phrasing used in technical regulations.
            USER QUESTION: "{orig_quest}"
            INSTRUCTIONS:
            1. TARGET ENTITY: 
            - Identify the specific part or parameter (e.g., "Wheelbase", "Main Hoop", "Impact Attenuator").
            - If a code like T.1.1 is provided, use it as the primary key.
            2. KEYWORDS ONLY:
            - Keep it minimal.
            - AVOID meta-words: "requirements", "specifications", "definition", "standard", "compliance".
            - AVOID conversational filler.
            3. SEARCH QUERY EXAMPLES:
            - Input: "How big should the wheelbase be?" -> Output: "Wheelbase minimum"
            - Input: "Tell me about rule T.3.2" -> Output: "Rule T.3.2"
            - Input: "What's the thickness of the main hoop?" -> Output: "Main hoop thickness"
            - Input: "Do we need a brake light?" -> Output: "Brake light"
            4. RAG_K SELECTION — how many rules to retrieve:
            - Use 5 (default): single fact questions ("what is the minimum wheelbase?", "is a brake light required?")
            - Use 10-15: questions asking for a group of rules on one topic ("what are the requirements for the roll hoop?")
            - Use 20-30: questions explicitly asking to LIST or ENUMERATE ALL rules under a condition
            OUTPUT FORMAT:
            Return ONLY a JSON object:
            {{
                "rag_search_query": "Your concise search query",
                "rag_k": <integer>
            }}
            CRITICAL: No explanations, no markdown. ONLY JSON.
            """

        response = ollama.generate(
            model=self.model, 
            prompt=prompt, 
            format="json",
            keep_alive=300, 
            options={"temperature": 0.15}
            )

        return response
    
    def analyze_drawing(self, rag_context, question, objects_str=None, original_image = None, YOLO_processed_image = None):

        system_prompt = "You are the Lead FSAE Technical Scrutineer. You do not think out loud. You only output the final requested format."

        if original_image is None:

            text_only_prompt = f"""
            **TECHNICAL CONTEXT (RAG):** {self.format_rag_context(rag_context)}
            ### USER QUESTION:
            "{question}"
            """
            
            response = ollama.generate(
                model=self.model,
                prompt=text_only_prompt,
                system=system_prompt,
                keep_alive=300,
                options={
                    "num_ctx": 32768,
                    "temperature": 0.0,
                    "top_p": 0.95,
                    "repeat_penalty": 1.1, 
                    "num_predict": 8192,
                    "think": False 
                }
            )
            

        elif original_image is not None and YOLO_processed_image is None:

            original_vision_prompt = f"""
            **TECHNICAL CONTEXT (RAG):** {self.format_rag_context(rag_context)}
            ### USER QUESTION:
            "{question}"
            """

            response = ollama.generate(
                model=self.model,
                prompt=original_vision_prompt,
                system=system_prompt,
                images=[original_image],
                keep_alive=300,
                options={
                    "num_ctx": 32768,
                    "temperature": 0.0,
                    "top_p": 0.95,
                    "repeat_penalty": 1.1, 
                    "num_predict": 8192,
                    "think": False 
                }
            )


        else:

            vision_prompt = f"""
            ### INPUT DATA:
            1. **TECHNICAL CONTEXT (RAG):** {self.format_rag_context(rag_context)}
            2. **ORIGINAL DRAWING:** [Provided Image]
            3. **YOLO DETECTIONS:** [Image with Bounding Boxes]
            You will be given a part of the drawing with the element that YOLO found on the image: {objects_str}
            Before this, another model labeled the data for you using YOLO.
            ### USER QUESTION:
            "{question}"
            """
            images_list = [img for img in [original_image, YOLO_processed_image] if img is not None]
            
            response = ollama.generate(
                model=self.model,
                prompt=vision_prompt,
                system=system_prompt,
                images=images_list,
                keep_alive=300,
                options={
                    "num_ctx": 32768,
                    "temperature": 0.0,
                    "top_p": 0.95,
                    "repeat_penalty": 1.1, 
                    "num_predict": 8192,
                    "think": False 
                          }
            )

        return self._get_response_text(response)
    
    def _get_response_text(self, response) -> str:
        text = getattr(response, "response", "") or ""

        if not text.strip():
            text = getattr(response, "thinking", "") or ""
        
        if not text.strip() and hasattr(response, "model_dump"):
            d = response.model_dump()
            text = d.get("response") or d.get("thinking") or ""
        
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        
        return text

In [4]:
class ScrutineerPipeline:
    def __init__(
            self, 
            pdf_path = r"AI_agent_FSAE\rules_pdf\FSAE_Rules_2024_V1.pdf",
            output_dir="experiment_results"
            ):

        self.retriever = FSAEHybridRAG(pdf_path)
        self.config_generator = DecesionCenter()
        self.output_dir = output_dir

        self.log_path = os.path.join(self.output_dir, "logs")
        os.makedirs(self.log_path, exist_ok=True)
        os.makedirs("yolov26_answers", exist_ok=True)
        os.makedirs("yolo_world_answers", exist_ok=True)

    def process_question(self, question, image_path, question_number):

        log_file = os.path.join(self.log_path, f"q_{question_number}.json")
        
        if os.path.exists(log_file):
            print(f" Вопрос №{question_number} уже обработан. Пропускаем.")
            with open(log_file, 'r', encoding='utf-8') as f:
                return json.load(f).get('final_answer', 'Ошибка чтения')
        
        run_data = {
            "question_number": question_number,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "input": {"question": question, "image_path": image_path},
            "intermediate_steps": {},
            "final_answer": None,
            "status": "started"
        }

        print(f"--- Старт обработки вопроса: '{question}' ---")
        
        print("Формирование вопроса для RAG ... ")
        raw_tasks = self.config_generator.database_query(question)
        thinking_json = json.loads(self._extract_json(raw_tasks))
        
        search_query = thinking_json['rag_search_query']
        rag_k = int(thinking_json.get('rag_k', 5))  
        print(f" Запрос: '{search_query}', rag_k={rag_k}")

        rag_search = self.retriever.get_context(search_query, k=rag_k)

        run_data["intermediate_steps"]["rag"] = {
                "search_query": search_query,
                "retrieved_context": rag_search,
                "rag_k": rag_k
            }
        
        if not self._is_valid_image_path(image_path):
            if image_path is not None:
                print(f" Путь к изображению '{image_path}' не найден — переходим в текстовый режим.")
            image_path = None

        if image_path is None:
            print(" Финальный анализ Gemma4...")
            final_answer = self.config_generator.analyze_drawing(rag_search, question)
            run_data["intermediate_steps"]["detection"] = "skipped (text-only)"
         

        else:
            print(" Генерация параметров детекции...")
            raw_config = self.config_generator.keywords_generate(question, rag_search)
            run_data["intermediate_steps"]["yolo_config_raw"] = self._extract_json(raw_config)

            config = None
            try:
                config = json.loads(self._extract_json(raw_config))
                run_data["intermediate_steps"]["yolo_config_parsed"] = config
            except Exception as e:
                print(f" Ошибка JSON: {e}")
                run_data["intermediate_steps"]["yolo_config_parsed"] = "Invalid JSON"

            if config:
                print("Запуск детекции...")
                try:
                    crop_path, objects_str = self._run_detection(image_path, config, question_number)
                    run_data["intermediate_steps"]["crop_image_path"] = crop_path
                    run_data["intermediate_steps"]["founded_objects"] = objects_str 
                    time.sleep(2) 

                    print("4. Финальный анализ Qwen3-VL (с YOLO)...")
                    final_answer = self.config_generator.analyze_drawing(
                        rag_context=rag_search, 
                        question=question,
                        objects_str=objects_str,
                        original_image=image_path,
                        YOLO_processed_image=crop_path
                        )

                except Exception as e:
                    print(f" ОШИБКА YOLO/SAHI: {e}")
                    final_answer = self.config_generator.analyze_drawing(
                        rag_context=rag_search, 
                        question=question, 
                        original_image=image_path
                        )
            else:
                # Fallback: если JSON кривой
                print("Финальный анализ Gemma4 (без YOLO, не валидный конфиг)...")
                final_answer = self.config_generator.analyze_drawing(
                    rag_context=rag_search, 
                    question=question, 
                    original_image=image_path
                    )

        
        run_data["final_answer"] = final_answer
        run_data["status"] = "success"

        log_file = os.path.join(self.log_path, f"q_{question_number}.json")
        with open(log_file, 'w', encoding='utf-8') as f:
            json.dump(run_data, f, ensure_ascii=False, indent=4)

        return final_answer
    
    def _run_detection(self, image_path, config, q_num):

        self._cleanup_vram() 
        gc.collect()
        time.sleep(1)

        if config['detection_model'].lower() == 'yolov26':

            model_raw = YOLO("yoloe-26x-seg.pt")

            model_raw.set_classes(config['keywords'])

            model_raw.save("fsae_yolov26_task.pt")

            detection_model = AutoDetectionModel.from_pretrained(
                model_type="ultralytics",
                model_path="fsae_yolov26_task.pt",
                confidence_threshold=0.35,
                device="cuda"
            )

            result = get_sliced_prediction(
                image_path,
                detection_model,
                slice_height=config['slice_height'],
                slice_width=config['slice_width'],
                overlap_height_ratio=config['overlap_height_ratio'],
                overlap_width_ratio=config['overlap_width_ratio'],
                postprocess_type="NMS",
                postprocess_match_threshold=0.05,
            )

            result.export_visuals(export_dir="yolov26_answers", file_name=f"yolov26_sahi_{q_num}")

            full_path = rf'AI_agent_FSAE\yolov26_answers\yolov26_sahi_{q_num}.png'

        elif 'world' in config['detection_model'].lower():

            model_raw = YOLOWorld("yolov8x-worldv2.pt")

            model_raw.set_classes(config['keywords'])

            model_raw.save("fsae_world_task.pt")

            detection_model = AutoDetectionModel.from_pretrained(
                model_type="yolov8",
                model_path="fsae_world_task.pt",
                confidence_threshold=0.35,
                device="cuda"
            )

            result = get_sliced_prediction(
                image_path,
                detection_model,
                slice_height=config['slice_height'],
                slice_width=config['slice_width'],
                overlap_height_ratio=config['overlap_height_ratio'],
                overlap_width_ratio=config['overlap_width_ratio'],
                postprocess_type="NMS",
                postprocess_match_threshold=0.05,
            )

            result.export_visuals(export_dir="yolo_world_answers", file_name=f"yoloworld_sahi_{q_num}")

            full_path = rf'AI_agent_FSAE\yolo_world_answers\yoloworld_sahi_{q_num}.png'

        predictions = result.object_prediction_list

        crop_path = None
        objects_str = None
        
        if len(predictions) > 0:
            img = PILImage.open(full_path)
            w, h = img.size

            found_objects = [p.category.name for p in result.object_prediction_list]
            objects_str = ", ".join(set(found_objects))
            
            x1_list = [p.bbox.minx for p in predictions]
            y1_list = [p.bbox.miny for p in predictions]
            x2_list = [p.bbox.maxx for p in predictions]
            y2_list = [p.bbox.maxy for p in predictions]
            
            min_x, min_y = min(x1_list), min(y1_list)
            max_x, max_y = max(x2_list), max(y2_list)
            
            pad_left = 450 
            pad_right = 100
            pad_top = 150  
            pad_bottom = 600
            
            final_box = (
                max(0, int(min_x - pad_left)),
                max(0, int(min_y - pad_top)),
                min(w, int(max_x + pad_right)),
                min(h, int(max_y + pad_bottom))
            )
            
            crop = img.crop(final_box)
            
            max_size = 2048
            if crop.width > max_size or crop.height > max_size:
                crop.thumbnail((max_size, max_size), PILImage.LANCZOS)
                
            crop_path = os.path.abspath(os.path.join(self.output_dir, f"q_{q_num}_focus.jpg"))
            crop.save(crop_path, "JPEG", quality=95)

        if 'model_raw' in locals(): del model_raw
        if 'detection_model' in locals(): del detection_model
        if 'result' in locals(): del result
        self._cleanup_vram()
        
        return crop_path, objects_str

    def _extract_json(self, ollama_dict):
        text = ollama_dict.get('response', '')
        if not text.strip():
            text = ollama_dict.get('thinking', '')
        
        text = text.replace('```json', '').replace('```', '').strip()
        return text
    
    def _is_valid_image_path(self, image_path) -> bool:
        if image_path is None:
            return False
        if isinstance(image_path, float):  
            return False
        path_str = str(image_path).strip()
        if path_str == '' or path_str.lower() == 'nan':
            return False
        return os.path.isfile(path_str)
    
    def _cleanup_vram(self):
        if torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

In [5]:
pipeline = ScrutineerPipeline()
questions_list = data_to_process.to_dict('records')

for i, row in enumerate(questions_list):
    try:
        pipeline.process_question(
            question=row["question"],
            image_path=row["image"],
            question_number=i + 1,
        )
    except KeyboardInterrupt:
        break

    except Exception as e:
        print(f"! Сбой на вопросе {i+1}: {e}")
        continue

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        time.sleep(3)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

--- Старт обработки вопроса: 'We are a student engineering team designing a vehicle for the FSAE competition. Attached is the FSAE rules document. Also attached is an image that shows an engineering drawing of the vehicle on the top accompanied by six CAD views of the vehicle on the bottom. The six CAD views each feature a different orientation of our design, so that 3D information about our design can be inferred. The CAD views are provided to contextualize the engineering drawing, which has the same orientation as one of the six CAD views. All units displayed in the engineering drawing have units of mm. Based on the engineering drawing, does our design comply with rule V.1.2 specified in the FSAE rule document? Only use dimensions explicitly shown in the engineering drawing to answer the question. If a dimension is not explicitly shown, you can assume that it complies with the rules. First provide an explanation for your answer (begin it with 'Explanation:'). Then provide just a yes/